# Lekcija 12 - Zmanjšanje zgodovine klepeta z agentovim delovnim prostorom

Ta zvezek prikazuje, kako upravljati s kontekstom v dolgih pogovorih z uporabo Microsoft Agent Framework. Ko pogovori rastejo, se število žetonov povečuje — sčasoma preseže okno konteksta modela. To rešimo z **vzorec povzetka konteksta** in **agentovim delovnim prostorom** za vztrajno pomnjenje.

## Kaj se boste naučili:
1. **Zakaj je upravljanje konteksta pomembno**: Razumevanje omejitev žetonov in oken konteksta
2. **Agentje, ki so ozaveščeni o kontekstu**: Gradnja agentov, ki upravljajo svoj lasten kontekst pogovora
3. **Vzorec povzetka konteksta**: Uporaba orodij za strnjenje zgodovine pogovora
4. **Agentov delovni prostor**: Vztrajno pomnjenje, ki preživi zmanjšanje konteksta

## Predpogoji:
- Nastavitev Azure OpenAI z konfiguriranimi okoljskimi spremenljivkami
- Razumevanje osnovnih konceptov agentov iz prejšnjih lekcij


## Namestitev


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
import asyncio
import dotenv
from datetime import datetime
from pathlib import Path

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

In [ ]:
dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## Zakaj je upravljanje konteksta pomembno

Vsak LLM ima omejeno **okno konteksta** — največje število tokenov, ki jih lahko obdeluje v enem zahtevku. Ko se pogovor z več krogi nadaljuje:

- **Število tokenov linearno narašča** z vsakim sporočilom uporabnika in odgovorom asistenta.
- **Tokeni poziva predstavljajo glavni strošek**, ker se celotna zgodovina vsakič ponovno pošlje.
- Na koncu pogovor **preseže okno konteksta** in model ga bodisi skrajša ali vrže napako.

### Strategije za upravljanje konteksta

| Strategija | Kako deluje | Kompromis |
|---|---|---|
| **Skrajšanje** | Izpusti najstarejša sporočila | Izguba zgodnjega konteksta |
| **Povzemanje** | Stisne starejša sporočila v povzetek | Nekateri detajli so izgubljeni, a ključne točke ostanejo |
| **Zapisovalnik / Zunanje pomnjenje** | Shrani ključne dejstvo izven pogovora | Zahteva klice orodij, a preživi vsako skrajšanje |

V tem zvezku kombiniramo **povzemanje** z orodjem za **zapisovalnik**, da agent lahko ohranja kontinuiteto tudi, kadar je zgodovina pogovora stisnjena.


## Ustvarjanje agenta, ki razume kontekst


In [ ]:
agent = client.as_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("Context-aware travel planning agent created")

## Simulacija dolgega pogovora

Raziščimo večkratni pogovor, da vidimo, kako se kontekst kopiči. Agent naj ohranja ključne podrobnosti (preferenc, proračun, datume potovanja) skozi več sprotnih pogovorov in pokaže kontinuiteto.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Opazite, kako agent ohranja kontekst iz prejšnjih krogov — ve o Japonski, sušiju, templjih, fotografiji, proračunu 3000 $, samostojnem potovanju in aprilski časovni okvir. V kratkem pogovoru to deluje dobro, vendar postane s povečevanjem pogovora polna zgodovina draga za ponovno pošiljanje.

Nadaljujmo pogovor z več krogi, da vidimo kopičenje konteksta:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Vzorec povzetka konteksta

Ko se pogovor razvija, lahko uporabimo **orodje za povzetke**, da zgostimo nabrani kontekst v kompaktno obliko. Agent uporablja to orodje za zapis ključnih preferenc, tako da tudi če starejša sporočila izbrišemo, so bistvene informacije shranjene.

Ta vzorec je gradnik za bolj sofisticirano zmanjševanje zgodovine:
1. Agent prepozna ključna dejstva iz pogovora
2. Pokliče orodje za povzetke, da jih shrani
3. Starejša sporočila je mogoče varno odstraniti, ker povzetek zajema bistvo

Spodaj definiramo orodje `summarize_preferences`, ki ga agent lahko uporabi za zapis kompaktnega povzetka naučenega.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = client.as_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Povzetek

V tej lekciji ste se naučili, kako upravljati s kontekstom v dolgotrajnih pogovorih agentov z uporabo Microsoft Agent Framework:

### Ključni pojmi
- **Okna konteksta so omejena** — vsak žeton v zgodovini pogovora stane denar in se šteje v omejitev.
- **Orodja za povzemanje** omogočajo agentu, da zgošči zbran kontekst v kompaktne povzetke, kar zmanjša uporabo žetonov ob ohranitvi bistvenih informacij.
- **Zvezki agentov** nudijo trajen zunanji pomnilnik, ki preživi katerokoli zmanjšanje pogovora.

### Kaj ste ustvarili
- **Agent, ki se zaveda konteksta**, ki ohranja kontinuiteto skozi večkratne pogovore
- **Orodje za povzemanje** (`summarize_preferences`), ki beleži ključne uporabniške podatke v kompaktni obliki
- **Večkratni pogovor**, ki prikazuje ohranjanje in obvladovanje sprememb konteksta

### Praktične uporabe
- **Pomoč strankam z boti**: Zapomnijo si preference med dolgimi podporniškimi sejami
- **Osebni asistenti**: Spremljajo tekoče projekte brez ponovnega razlaganja konteksta
- **Izobraževalni tutorji**: Ohranjajo napredek učencev skozi številne interakcije

### Naslednji koraki
- Implementirajte polno orodje za zvezke z vztrajanjem podatkov na datotekah
- Dodajte samodejno krajšanje zgodovine po povzetku
- Združite s podatkovnimi zbirkami vektorjev za semantično iskanje pomnilnika
- Ustvarite agente, ki lahko nadaljujejo pogovore dneve kasneje s celotnim kontekstom


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Omejitev odgovornosti**:
Ta dokument je bil preveden z uporabo AI prevajalske storitve [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, vas prosimo, da upoštevate, da avtomatizirani prevodi lahko vsebujejo napake ali netočnosti. Izvirni dokument v njegovem izvirnem jeziku je treba obravnavati kot avtoritativni vir. Za kritične informacije je priporočljiv strokovni človeški prevod. Ne odgovarjamo za morebitna nesporazume ali napačne interpretacije, ki izhajajo iz uporabe tega prevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
